# Baseline hole clustering

This notebook consumes a feature matrix produced by `hylogger prepare --selection <name>`. It contains no acquisition or cleaning logic.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from hylogger.config import AppConfig

In [ ]:
SELECTION = "pilot"
N_CLUSTERS = 5
RANDOM_STATE = 42

config = AppConfig.load()
feature_path = config.data_dir / "prepared" / "features" / f"selection_{SELECTION}.parquet"
if not feature_path.exists():
    raise FileNotFoundError(f"Run: hylogger prepare --selection {SELECTION}\nMissing: {feature_path}")

features = pd.read_parquet(feature_path)
features.head()

In [ ]:
hole_ids = features["hole_id"].copy()
X = features.drop(columns=["hole_id", "prepared_at"], errors="ignore").select_dtypes(include="number")
if len(X) < N_CLUSTERS:
    raise ValueError(f"Need at least {N_CLUSTERS} prepared holes; found {len(X)}")

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20),
)
labels = model.fit_predict(X)
transformed = model[:-1].transform(X)
score = silhouette_score(transformed, labels)
print(f"Silhouette score: {score:.3f}")

In [ ]:
assignments = pd.DataFrame({"hole_id": hole_ids, "cluster_id": labels})
assignments.sort_values(["cluster_id", "hole_id"])

## Interpretation checklist

Before treating a cluster as geological, compare it with location, operator, scan generation, available spectral regions, processing level, mask rate, depth coverage, and raw-package dimensions.